In [0]:
%sql
CREATE VOLUME IF NOT EXISTS puc_data_specialist_de_2026_09.`00-raw`.`brazil-ecommerce`


In [0]:
# Install kaggle API (only needed once)
%pip install kaggle
dbutils.library.restartPython()


> **__NOTE__**
> I tried to setup a secret into Databricks through UI and CLI as per DB docs but UI secret configuration is only available when using dedicated clusters and/or when integrated with Azure Secrets. 
> As the import of the raw data is not part of the requirements, I imported all data into a volume and started from there. 

To setup a secret via the Databricks CLI:

```bash
# 1. Create a secret scope
databricks secrets create-scope kaggle-scope

# 2. Store the Kaggle API token as a secret
databricks secrets put-secret kaggle-scope kaggle_key --string-value "YOUR_KAGGLE_API_TOKEN"

# 3. Verify the scope and secret exist (values are never returned)
databricks secrets list-scopes --output JSON
databricks secrets list-secrets kaggle-scope --output JSON
```

Then in a notebook, retrieve the secret with:

```python
dbutils.secrets.get(scope="kaggle-scope", key="kaggle_key")
```

> **Note:** Secret-based authentication requires a dedicated cluster or Azure Key Vault integration. Serverless compute does not support `dbutils.secrets` the same way, so credentials were hardcoded for the data import step only.

In [0]:

# --- Creating a folder with a date information to prevent overwriting ---
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d")

volume_path = f"/Volumes/puc_data_specialist_de_2026_09/00-raw/brazil-ecommerce/{timestamp}"

# --- Configure Kaggle credentials ---
# Set your Kaggle username and API key here (from kaggle.com -> Account -> Create New Token)
KAGGLE_API_TOKEN="[ADD HERE THE API TOKEN FROM YOUR ACCOUNT]"

import os
os.environ["KAGGLE_USERNAME"] = "ADD HERE YOUR KAGGLE USERNAME FROM YOUR ACCOUNT"
os.environ["KAGGLE_KEY"] = KAGGLE_API_TOKEN

# --- Download dataset into the volume (no changes to files) ---
from kaggle import KaggleApi

api = KaggleApi()
api.authenticate()

api.dataset_download_files(
    "olistbr/brazilian-ecommerce",
    path=volume_path,
    unzip=True,        
    quiet=False,
)
